In [1]:
import pandas as pd
import numpy as np
from gensim.models.phrases import Phrases, Phraser
from gensim.models import Word2Vec
from gensim.models import KeyedVectors
from time import time 
import multiprocessing
from tqdm import tqdm
tqdm.pandas()

C:\Libraries\anaconda3\lib\site-packages\tqdm\std.py:697: FutureWarning: The Panel class is removed from pandas. Accessing it from the top-level namespace will also be removed in the next version
  from pandas import Panel


In [2]:
df = pd.read_csv("./sentiment140-cleaned.csv", encoding="ISO-8859-1", engine="python")
df['text'] = df.progress_apply(lambda row: row['text'].split(), axis=1)

100%|██████████| 1600000/1600000 [00:21<00:00, 74953.91it/s] 


In [3]:
sent = [row for row in tqdm(df['text'])]
phrases = Phrases(sent, min_count=1, progress_per=50000)
bigram = Phraser(phrases)
sentences = bigram[sent]
sentences[1]

['is',
 'upset',
 'that',
 'he',
 'can_t',
 'update',
 'his',
 'facebook',
 'by',
 'texting',
 'it',
 'and',
 'might',
 'cry',
 'as',
 'a',
 'result',
 'school',
 'today',
 'also',
 'blah',
 '!']

In [24]:
n = 10000
dfsample = df.sample(n=n, random_state=231)
cluster_good = ["good", "nice", "cool", "lovely", "wonderful", "great", "awesome", "fantastic", "amazing", "fun", "excellent"]
cluster_bad = ["bad", "horrible", "terrible", "awful", "worst", "shitty", "crappy", "sucks", "hate"]
def clusterDistance(word, cluster, word_vectors):
    return np.average(word_vectors.distances(word, cluster))

def predictTweet(text, word_vectors):
    wordvectors = []
    for word in text:
        if not (word in word_vectors.vocab):
            continue
        wordvectors.append(word_vectors[word])
    if len(wordvectors) == 0:
        return 0
    sentencevector = np.mean(wordvectors, axis=0)

    positive = clusterDistance(sentencevector, cluster_good, word_vectors)
    negative = clusterDistance(sentencevector, cluster_bad, word_vectors)
    if positive < negative:
        return 4
    return 0

In [32]:
def testw2v(alpha, window, min_count, min_alpha, sample):
    w2v_model = Word2Vec(
        min_count=min_count,
        window=window,
        size=300,
        sample=sample,
        alpha=alpha,
        min_alpha=min_alpha,
        negative=20,
        seed=231,
        workers=multiprocessing.cpu_count()-1)

    w2v_model.build_vocab(sentences, progress_per=50000)
    print("Vocab built")
    w2v_model.train(sentences, total_examples=w2v_model.corpus_count, epochs=30, report_delay=1)
    print("Model trained")
    w2v_model.init_sims(replace=True)
    word_vectors = w2v_model.wv
    dfsample['predict'] = dfsample.progress_apply(lambda row: predictTweet(row['text'], word_vectors), axis=1)
    dfsample['predict_correct'] = dfsample.progress_apply(lambda row: row['sentiment'] == row['predict'], axis=1)
    accuracy = dfsample['predict_correct'].value_counts()[True] / n
    print("\n")
    print(accuracy)

In [31]:
#default     alpha = 0.025, window = 5, min_count=5, min_alpha = 0.0001 sample=0.001
#out default alpha = 0.03,  window = 4, min_count=3, min_alpha = 0.0007 sample=1e-5
testw2v(0.03, 4, 3, 0.0007, 1e-5)

Vocab built
100%|██████████| 10000/10000 [00:00<00:00, 105523.19it/s]


0.5281


In [29]:
testw2v(0.03, 4, 3, 0.0007, 0.001)

Vocab built
100%|██████████| 10000/10000 [00:00<00:00, 107241.58it/s]


0.6327


In [33]:
testw2v(0.03, 4, 3, 0.0001, 1e-5)

Vocab built
100%|██████████| 10000/10000 [00:00<00:00, 104462.73it/s]


0.6888


In [ ]:
testw2v(0.03, 4, 5, 0.0007, 1e-5)

In [ ]:
testw2v(0.03, 5, 3, 0.0007, 1e-5)

In [ ]:
testw2v(0.025, 4, 3, 0.0007, 1e-5)